# Jon2 AI — Private Arabic Corpus Audit
This notebook audits your private JSONL files inside Google Colab. It downloads only the audit script from the Jon2 GitHub branch; it does **not** upload your dataset to GitHub or print conversation text.

It searches Google Drive for `jon2_dialogue_validated.jsonl`. If separate train/test JSONL files with clear names exist, it uses them. Otherwise it creates a deterministic 80/20 split in the temporary Colab runtime and audits that split.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
matches = list(Path('/content/drive/MyDrive').rglob('jon2_dialogue_validated.jsonl'))
print('Matching validated corpus files:', len(matches))
for p in matches[:10]:
    print(f'{p} — {p.stat().st_size:,} bytes')
if len(matches) != 1:
    raise RuntimeError('Expected exactly one jon2_dialogue_validated.jsonl. No dataset contents were displayed. Resolve duplicate/missing files before continuing.')
SOURCE_JSONL = matches[0]


In [ ]:
import json, hashlib, random, sys, urllib.request
from pathlib import Path

repo_branch = 'feature/jon2-from-scratch'
audit_url = f'https://raw.githubusercontent.com/jonkabeel-star/jon2-ai/{repo_branch}/ai_engine/scratch/audit_corpus.py'
audit_path = Path('/content/audit_corpus.py')
urllib.request.urlretrieve(audit_url, audit_path)
print('Downloaded the public audit utility. Private data remains in your Drive/Colab runtime.')

# Prefer an already separated pair when the filenames are unambiguous.
root = SOURCE_JSONL.parent
train_candidates = sorted(root.glob('*train*.jsonl'))
test_candidates = sorted(root.glob('*test*.jsonl'))
train_candidates = [p for p in train_candidates if p != SOURCE_JSONL]
test_candidates = [p for p in test_candidates if p != SOURCE_JSONL]

if len(train_candidates) == 1 and len(test_candidates) == 1:
    TRAIN_JSONL, TEST_JSONL = train_candidates[0], test_candidates[0]
    print('Using existing train/test files by filename.')
else:
    rows = []
    with SOURCE_JSONL.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                rows.append((line_no, json.loads(line)))
    # Stable split: same input produces the same partition every time.
    rows.sort(key=lambda item: hashlib.sha256(json.dumps(item[1], ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest())
    test_count = max(1, round(len(rows) * 0.2))
    test_rows, train_rows = rows[:test_count], rows[test_count:]
    out_dir = Path('/content/jon2_private_audit_split')
    out_dir.mkdir(exist_ok=True)
    TRAIN_JSONL, TEST_JSONL = out_dir/'jon2_train.jsonl', out_dir/'jon2_test.jsonl'
    for path, subset in ((TRAIN_JSONL, train_rows), (TEST_JSONL, test_rows)):
        with path.open('w', encoding='utf-8') as f:
            for _, obj in subset:
                f.write(json.dumps(obj, ensure_ascii=False) + '\\n')
    print(f'Created deterministic temporary split: {len(train_rows)} train / {len(test_rows)} test records.')
    print('The original Drive file was not changed.')

print('Train file:', TRAIN_JSONL.name)
print('Test file:', TEST_JSONL.name)


In [ ]:
import runpy
sys.argv = ['audit_corpus.py', '--train', str(TRAIN_JSONL), '--test', str(TEST_JSONL)]
runpy.run_path(str(audit_path), run_name='__main__')


## Interpreting the result
- `training_ready: true` means only that the configured automated data checks passed. It does not guarantee good model quality or legal permission.
- `training_ready: false` means do not start training yet. Review the failed checks and resolve them using genuinely new, permitted, reviewed data.
- The current minimum is 100,000 training characters and 100 training records. Do not lower it just to force training to start.
- Keep the test split untouched. Never train on it.
- Do not upload private JSONL files or model checkpoints to the public GitHub repository.